# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [70]:
# imports
import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import ollama

In [71]:
# constants
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [72]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
openai = OpenAI()

API key looks good so far


In [73]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""
system_prompt = """
                    You are a helpful assistant. 
                    Respond only with a valid JSON object containing relevant links.
               """

In [74]:
print(system_prompt)


                    You are a helpful assistant. 
                    Respond only with a valid JSON object containing relevant links.
               


In [75]:
def get_user_prompt(question):
    user_prompt = f"Here is the answer to the question {question} - "
    return user_prompt

In [76]:
print(get_user_prompt(question))

Here is the answer to the question 
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
 - 


In [77]:
# Get gpt-4o-mini to answer, with streaming - My solution

def get_stream(question):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_user_prompt(question)}
        ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        # Safely check if this chunk contains content
        if hasattr(chunk.choices[0].delta, "content") and chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content
            response = response.replace("```", "").replace("markdown", "")
            update_display(Markdown(response), display_id=display_handle.display_id)

    return response



In [78]:
system_prompt = "You are a helpful technical tutor who answers questions about python code, software engineering, data science and LLMs"
user_prompt = "Please give a detailed explanation to the following question: " + question
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]
stream = openai.chat.completions.create(model=MODEL_GPT, messages=messages,stream=True)
    
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    response = response.replace("```","").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)

Sure! Let’s break down the given line of code step by step to understand what it does and why.

### Code Explanation

python
yield from {book.get("author") for book in books if book.get("author")}


1. **Understanding `yield from`:**
   - The `yield from` expression is used in Python to delegate part of a generator's operations to another generator. This allows you to iterate over items from a sub-iterator or something similar smoothly. It effectively allows you to yield values from an iterable as if they were being yielded from the parent generator.

2. **Set Comprehension:**
   - The part `{book.get("author") for book in books if book.get("author")}` is a **set comprehension**. Set comprehensions are a syntactic construct for creating sets in Python comprehensively.
   - This specific set comprehension iterates over `books`, which is expected to be an iterable (like a list of dictionaries), where each dictionary represents a book.

3. **Accessing Dictionary Values:**
   - `book.get("author")` attempts to access the value associated with the key `"author"` in each `book` dictionary.
   - The method `.get(key)` retrieves the value for `key` if it exists, or returns `None` (or a specified default if provided). This is useful for safely accessing dictionary items without raising KeyErrors.

4. **Filtering Non-None Values:**
   - The `if book.get("author")` condition is a filter that ensures only non-None (or truthy) author names are included in the resulting set. This means it will skip any books that do not have an author (i.e., where `book.get("author")` returns `None`, an empty string, or any other falsey value).

5. **Creating a Set:**
   - The result of the comprehension—the authors' names—are collected into a set. Using a set automatically handles duplicates since sets do not allow duplicate entries. So if two books have the same author, this will only keep one entry for that author.

### Why This Code is Written This Way

1. **Efficiency in Handling Duplicates:**
   - By using a set, the code efficiently collects unique authors from the list of books. If the goal is to have only distinct authors, a set is a perfect choice for this purpose.

2. **Avoiding Errors:**
   - By using `.get()` to access the "author" key, the code avoids potential `KeyError` exceptions that would occur if a key is missing. This adds robustness, especially when dealing with data that might not be perfectly structured.

3. **Generator Yielding:**
   - The `yield from` construct allows this code to yield each author name back to the caller of the generator without needing to construct an intermediate list or set explicitly. It makes the function more memory efficient and allows for processing large datasets without loading everything into memory at once.

### Example Scenario

Here's an example to illustrate how this might work:

python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3"},  # No author
    {"title": "Book 4", "author": "Author A"}
]

# Hypothetical generator function containing the yield from line
def unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Using the generator
for author in unique_authors(books):
    print(author)


### Output:

Author A
Author B


In this example, even though there are two books by "Author A", the output only shows each author once thanks to the set comprehension.

### Conclusion
In summary, the provided code collects unique author names from a list of books, handles missing or non-existent author keys gracefully, and uses generator yielding to provide an efficient way to traverse these authors.

In [79]:
get_stream(question)

The code snippet you've provided is using a generator expression along with the `yield from` statement to produce a sequence of authors from a collection of books.

Let's break it down:

### Explanation of the Code:
1. **Set Comprehension**:
   python
   {book.get("author") for book in books if book.get("author")}
   
   - This part creates a set of authors from the `books` collection.
   - It iterates over each `book` in the `books` iterable (which is expected to be a list or collection of dictionaries).
   - For each `book`, it calls `book.get("author")` to retrieve the author of the book, ensuring that the operation is safe (it won't raise an error if "author" key doesn't exist; instead, it will return `None`).
   - The condition `if book.get("author")` filters out any books that do not have an author (i.e., it only includes books where the author is not `None` or an empty string).

2. **Yielding Results**:
   python
   yield from ...
   
   - `yield from` allows one generator to delegate part of its operations to another generator. It effectively "yields" every item from the iterable that follows it.
   - In this case, it yields each unique author from the set comprehension that was just created.

### Final Purpose:
Overall, the code snippet generates a sequence of unique authors from the `books` list where the authors' values are valid (not `None` or falsy). This is useful if you want to list all authors without duplicates.

### Example:
Assuming you have the following `books` list:
python
books = [
    {"title": "Book A", "author": "Author 1"},
    {"title": "Book B", "author": "Author 2"},
    {"title": "Book C", "author": None},
    {"title": "Book D", "author": "Author 1"},
    {"title": "Book E", "author": ""},
]

The code will yield:
- "Author 1"
- "Author 2"

### Conclusion:
This code effectively collects and yields unique author names from a list of books, discarding any entries that do not have an author specified. It is a concise and efficient way to gather this information while using Python's powerful generator and comprehension features.

'The code snippet you\'ve provided is using a generator expression along with the `yield from` statement to produce a sequence of authors from a collection of books.\n\nLet\'s break it down:\n\n### Explanation of the Code:\n1. **Set Comprehension**:\n   python\n   {book.get("author") for book in books if book.get("author")}\n   \n   - This part creates a set of authors from the `books` collection.\n   - It iterates over each `book` in the `books` iterable (which is expected to be a list or collection of dictionaries).\n   - For each `book`, it calls `book.get("author")` to retrieve the author of the book, ensuring that the operation is safe (it won\'t raise an error if "author" key doesn\'t exist; instead, it will return `None`).\n   - The condition `if book.get("author")` filters out any books that do not have an author (i.e., it only includes books where the author is not `None` or an empty string).\n\n2. **Yielding Results**:\n   python\n   yield from ...\n   \n   - `yield from` all

In [83]:
# Get Llama 3.2 to answer

response = ollama.chat(model=MODEL_LLAMA, messages=messages)
reply = response['message']['content']
display(Markdown(reply))


**Code Explanation**

This line of Python code uses several advanced features, including:

1.  **List Comprehension**: The `{}` syntax creates a new iterable that contains the result of the expression inside.
2.  **`yield from`**: This keyword is used to delegate part of the execution of `yield` statements from a generator function (the one containing `yield`) to another generator or iterator.
3.  **Dictionary iteration**: The `.get("author") for book in books if book.get("author")` part iterates over a dictionary (`books`) and filters it based on whether each value associated with the key `"author"` is present.

Let's break this down step by step:

-   `book.get("author")`: This calls the `get()` method of the `book` object (which should be a dictionary) with the key `"author"`. The `get()` method returns the value associated with the given key, and if that key is not present in the dictionary, it returns a default value (in this case, `None`). In the context of this list comprehension, `None` will be skipped over.

-   `for book in books`: This part starts an iteration over the `books` variable, which should contain one or more dictionaries. The loop will execute for each dictionary (`book`) in `books`.

-   `if book.get("author")`: This filters out any dictionaries that do not have a value associated with the key `"author"`.

Here's what happens when you put it all together:

1.  Create an empty list (`{}`).
2.  Iterate over each dictionary (`book`) in `books`.
3.  For each dictionary, iterate over its values (i.e., its "authors") that are associated with the key `"author"`.
4.  Add each author's value to the list.
5.  Once all authors have been found for a book, skip to the next book.
6.  Once all books have been processed, stop iterating.

**Example Use Case**

Suppose you're writing a program that processes a list of book objects, and you need to extract a list of authors from each book.

```python
# Assuming 'books' is a list of dictionaries where each dictionary represents a book

def get_authors(books):
    return {book.get("author") for book in books if book.get("author")}

# Create some sample data:
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "authors": ["Author B", "Author C"]},
    {"title": "Book 3"}
]

# Call the function
authors = get_authors(books)

print(authors)  # Output: {'Author A', 'Author B', 'Author C'}
```

This is a very concise way to perform this operation. It's especially useful when you're working with large datasets and want to avoid creating unnecessary intermediate data structures.

**Alternative Solution**

Here's an equivalent solution using traditional for loops:

```python
def get_authors(books):
    authors = []
    for book in books:
        if "author" in book:
            author = book["author"]
            # Handle cases where 'author' is a list of authors
            if isinstance(author, list):
                authors.extend(author)
            else:
                authors.append(author)
    return authors
```

However, the original solution using list comprehension and `yield from` is more concise and can be more efficient for large datasets.